# Doctor's Consultation Fee Analysis — Data Cleaning

### Purpose
This notebook cleans the historical Practo dataset and prepares the analysis-ready dataset used by the EDA stage.

### Data flow
```text
data/raw/practo_raw_data.csv
            ↓
      Data Quality Audit
            ↓
      Data Cleaning
            ↓
data/processed/practo_cleaned_data.csv
            ↓
 Column Renaming + Feature Engineering
            ↓
data/processed/practo_analysis_data.csv
```

### Historical dataset
- **Source:** Practo
- **Records:** 1,080
- **Initial columns:** 8
- **Final analysis columns:** 9
- Missing patient ratings are handled using **median imputation**.
- `Free Consultation` is represented as **₹0**.
- Consultation-fee and patient-rating outliers are **detected but retained** because they are treated as potentially valid observations.
- `City_Tier` is added during the final feature-preparation stage.

> **Important:** This notebook starts from the raw/uncleaned dataset. It does not use the final analysis dataset as its input.


## 1. Import Libraries

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


## 2. Define Project Paths

In [2]:
# Notebook location: project_root/notebooks/02_data_cleaning.ipynb
PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "practo_raw_data.csv"
CLEANED_PATH = PROJECT_ROOT / "data" / "processed" / "practo_cleaned_data.csv"

print("Raw dataset :", RAW_PATH)
print("Cleaned data:", CLEANED_PATH)

Raw dataset : C:\Users\ahmad\Doctor's_Consultation_Fee_Analysis\data\raw\practo_raw_data.csv
Cleaned data: C:\Users\ahmad\Doctor's_Consultation_Fee_Analysis\data\processed\practo_cleaned_data.csv


## 3. Load the Raw Dataset

The raw dataset contains the original scraped/combined values before cleaning.

Expected raw columns:

`Doctor_Name`, `Specialization`, `Exprience`, `Hospital_Name`, `City`, `Patient_Like`, `Info_Source`, `Consultation_Fee`


In [3]:
df = pd.read_csv(RAW_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


Dataset shape: 1,080 rows × 8 columns


,Doctor_Name,Specialization,Exprience,Hospital_Name,City,Patient_Like,Info_Source,Consultation_Fee
0,Dr. Anil Krishna Gundala,Cardiologist,23,Medicover Hospitals - Hitech City,Hyderabad,96%,Practo,₹800
1,Dr. Sai Ravi Shanker A,Cardiologist,31,Aster Prime Hospital,Hyderabad,60%,Practo,₹800
2,Dr. R. Balaji,Cardiologist,37,Medicover Hospitals - Hitech City,Hyderabad,97%,Practo,₹900
3,Dr. Krishna Prasad A R,Cardiologist,32,Medicover Hospitals - Hitech City,Hyderabad,82%,Practo,₹800
4,Dr. Shiva Prasad Koyalakonda,Cardiologist,23,One Cardiac Centre,Hyderabad,93%,Practo,₹700


## 4. Initial Data Quality Audit

In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nUnique values in Patient_Like (sample):")
print(df["Patient_Like"].dropna().unique()[:15])

print("\nFree Consultation records:")
print((df["Consultation_Fee"] == "Free Consultation").sum())


Column names:
['Doctor_Name', 'Specialization', 'Exprience', 'Hospital_Name', 'City', 'Patient_Like', 'Info_Source', 'Consultation_Fee']

Data types:


,dtype
Doctor_Name,object
Specialization,object
Exprience,int64
Hospital_Name,object
City,object
Patient_Like,object
Info_Source,object
Consultation_Fee,object



Missing values:


,missing_values
Doctor_Name,0
Specialization,0
Exprience,0
Hospital_Name,0
City,0
Patient_Like,141
Info_Source,0
Consultation_Fee,0



Duplicate rows:
0

Unique values in Patient_Like (sample):
['96%' '60%' '97%' '82%' '93%' '98%' '100%' '95%' '89%' '91%' '75%' '86%'
 '83%' '33%' '50%']

Free Consultation records:
27


## 5. Clean Patient Ratings

`Patient_Like` contains percentage strings such as `96%` and missing values.

Cleaning steps:
1. Remove `%`.
2. Convert the column to numeric.
3. Calculate the median from available ratings.
4. Fill missing ratings with the median.


In [5]:
df["Patient_Like"] = df["Patient_Like"].str.replace("%", "", regex=False)
df["Patient_Like"] = pd.to_numeric(df["Patient_Like"], errors="coerce")

rating_median = df["Patient_Like"].median()

print(f"Median patient rating used for imputation: {rating_median:.2f}%")
print(f"Missing ratings before imputation: {df['Patient_Like'].isna().sum()}")

df["Patient_Like"] = df["Patient_Like"].fillna(rating_median)

print(f"Missing ratings after imputation: {df['Patient_Like'].isna().sum()}")


Median patient rating used for imputation: 95.00%
Missing ratings before imputation: 141
Missing ratings after imputation: 0


## 6. Clean Consultation Fees

The consultation-fee field contains currency-formatted strings and `Free Consultation`.

Cleaning rules:
- `Free Consultation` → `₹0`
- Remove the `₹` symbol
- Convert the result to integer


In [6]:
free_consultation_count = (df["Consultation_Fee"] == "Free Consultation").sum()

df["Consultation_Fee"] = df["Consultation_Fee"].replace(
    "Free Consultation", "₹0"
)

df["Consultation_Fee"] = (
    df["Consultation_Fee"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.strip()
)

df["Consultation_Fee"] = pd.to_numeric(
    df["Consultation_Fee"], errors="raise"
).astype(int)

print(f"Free Consultation records converted to ₹0: {free_consultation_count}")
print(f"Consultation fee dtype: {df['Consultation_Fee'].dtype}")


Free Consultation records converted to ₹0: 27
Consultation fee dtype: int64


## 7. Standardize Hospital Names

The original dataset contains hospital names with location suffixes separated by a hyphen, for example:

`Medicover Hospitals - Hitech City`

The historical cleaning workflow keeps the hospital name before the first hyphen.


In [7]:
hyphenated_hospitals = df["Hospital_Name"].astype(str).str.contains("-", regex=False).sum()

df["Hospital_Name"] = df["Hospital_Name"].astype(str).str.split("-").str[0].str.strip()

print(f"Hospital names containing a hyphen before cleaning: {hyphenated_hospitals}")


Hospital names containing a hyphen before cleaning: 74


## 8. Check Duplicate Records

In [8]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

if duplicate_count == 0:
    print("No duplicate rows were found; no records were removed.")
else:
    df = df.drop_duplicates().copy()
    print(f"Removed {duplicate_count} duplicate rows.")


Duplicate rows: 0
No duplicate rows were found; no records were removed.


## 9. Detect Outliers — Do Not Automatically Remove Them

Outliers are checked using the IQR method for:
- Consultation Fee
- Patient Rating

The original workflow **detected these observations but retained them**. Therefore, no outlier rows are removed in the cleaning stage.

The EDA notebook may create a separate filtered view for sensitivity analysis.


In [9]:
def iqr_outlier_summary(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)

    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Outlier Count": int(mask.sum()),
    }

fee_outliers = iqr_outlier_summary(df["Consultation_Fee"])
rating_outliers = iqr_outlier_summary(df["Patient_Like"])

outlier_summary = pd.DataFrame(
    [fee_outliers, rating_outliers],
    index=["Consultation_Fee", "Patient_Like"]
)

display(outlier_summary)
print("Outliers are retained in the cleaned dataset.")


,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
Consultation_Fee,500.0,1000.0,500.0,-250.0,1750.0,39
Patient_Like,89.0,100.0,11.0,72.5,116.5,98


Outliers are retained in the cleaned dataset.


## 10. Validate the Cleaned Dataset

In [10]:
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDuplicate rows:")
print(df.duplicated().sum())

display(df.head())


Shape: 1,080 rows × 8 columns

Missing values:


,missing_values
Doctor_Name,0
Specialization,0
Exprience,0
Hospital_Name,0
City,0
Patient_Like,0
Info_Source,0
Consultation_Fee,0



Data types:


,dtype
Doctor_Name,object
Specialization,object
Exprience,int64
Hospital_Name,object
City,object
Patient_Like,float64
Info_Source,object
Consultation_Fee,int64



Duplicate rows:
0


,Doctor_Name,Specialization,Exprience,Hospital_Name,City,Patient_Like,Info_Source,Consultation_Fee
0,Dr. Anil Krishna Gundala,Cardiologist,23,Medicover Hospitals,Hyderabad,96.0,Practo,800
1,Dr. Sai Ravi Shanker A,Cardiologist,31,Aster Prime Hospital,Hyderabad,60.0,Practo,800
2,Dr. R. Balaji,Cardiologist,37,Medicover Hospitals,Hyderabad,97.0,Practo,900
3,Dr. Krishna Prasad A R,Cardiologist,32,Medicover Hospitals,Hyderabad,82.0,Practo,800
4,Dr. Shiva Prasad Koyalakonda,Cardiologist,23,One Cardiac Centre,Hyderabad,93.0,Practo,700


## 11. Save the Cleaned Dataset

This file contains cleaned values but keeps the original column names used during the cleaning stage.

**Output:** `data/processed/practo_cleaned_data.csv`


In [11]:
df.to_csv(CLEANED_PATH, index=False)

print(f"Saved cleaned dataset to: {CLEANED_PATH}")


Saved cleaned dataset to: C:\Users\ahmad\Doctor's_Consultation_Fee_Analysis\data\processed\practo_cleaned_data.csv


## 12. Prepare the Analysis Dataset

The original project performed column renaming and added `City_Tier` after cleaning.

This stage converts the cleaned dataset into the canonical analysis dataset used by EDA.


In [12]:
ANALYSIS_PATH = (
    PROJECT_ROOT / "data" / "processed" / "practo_analysis_data.csv"
)

In [13]:
analysis_df = df.copy()

# City classification used in the original project
metro_cities = [
    "Hyderabad",
    "Bangalore",
    "Chennai",
    "Kolkata",
    "Mumbai",
    "Delhi",
]

analysis_df["City_Tier"] = np.where(
    analysis_df["City"].isin(metro_cities),
    "Metro",
    "Non-Metro",
)

analysis_df.rename(
    columns={
        "Patient_Like": "Patient_Rating(%)",
        "Info_Source": "Source",
        "Consultation_Fee": "Consultation_Fee(₹)",
        "Exprience": "Experience",
    },
    inplace=True,
)

# Place City_Tier immediately after City
columns = analysis_df.columns.tolist()
columns.remove("City_Tier")
city_index = columns.index("City") + 1
columns.insert(city_index, "City_Tier")

analysis_df = analysis_df[columns]

analysis_df.head()


,Doctor_Name,Specialization,Experience,Hospital_Name,City,City_Tier,Patient_Rating(%),Source,Consultation_Fee(₹)
0,Dr. Anil Krishna Gundala,Cardiologist,23,Medicover Hospitals,Hyderabad,Metro,96.0,Practo,800
1,Dr. Sai Ravi Shanker A,Cardiologist,31,Aster Prime Hospital,Hyderabad,Metro,60.0,Practo,800
2,Dr. R. Balaji,Cardiologist,37,Medicover Hospitals,Hyderabad,Metro,97.0,Practo,900
3,Dr. Krishna Prasad A R,Cardiologist,32,Medicover Hospitals,Hyderabad,Metro,82.0,Practo,800
4,Dr. Shiva Prasad Koyalakonda,Cardiologist,23,One Cardiac Centre,Hyderabad,Metro,93.0,Practo,700


## 13. Final Analysis Dataset Validation

In [14]:
expected_columns = [
    "Doctor_Name",
    "Specialization",
    "Experience",
    "Hospital_Name",
    "City",
    "City_Tier",
    "Patient_Rating(%)",
    "Source",
    "Consultation_Fee(₹)",
]

assert analysis_df.shape == (1080, 9), (
    f"Unexpected shape: {analysis_df.shape}"
)
assert analysis_df.columns.tolist() == expected_columns
assert analysis_df.isna().sum().sum() == 0
assert analysis_df.duplicated().sum() == 0

print("Final validation passed.")
print(f"Rows   : {len(analysis_df):,}")
print(f"Columns: {analysis_df.shape[1]}")
print("Missing values: 0")
print("Duplicate rows: 0")


Final validation passed.
Rows   : 1,080
Columns: 9
Missing values: 0
Duplicate rows: 0


## 14. Save the Analysis-Ready Dataset

In [15]:
analysis_df.to_csv(ANALYSIS_PATH, index=False)

print(f"Saved analysis-ready dataset to: {ANALYSIS_PATH}")


Saved analysis-ready dataset to: C:\Users\ahmad\Doctor's_Consultation_Fee_Analysis\data\processed\practo_analysis_data.csv


## 15. Cleaning Summary

| Stage | Transformation |
|---|---|
| Patient rating | Removed `%`, converted to numeric |
| Missing rating | Median imputation |
| Consultation fee | Removed `₹`, converted to numeric |
| Free consultation | Converted to ₹0 |
| Hospital name | Removed suffix after first `-` |
| Duplicates | Checked; none found |
| Outliers | Detected using IQR; retained |
| Column names | Standardized after cleaning |
| City classification | Added `City_Tier` |
| Final dataset | 1,080 rows × 9 columns |

### Output files

```text
data/
├── raw/
│   └── practo_raw_data.csv
│
└── processed/
    ├── practo_cleaned_data.csv
    └── practo_analysis_data.csv
```

The next notebook, **`03_exploratory_data_analysis.ipynb`**, should load:

```python
pd.read_csv("../data/processed/practo_analysis_data.csv")
```
